# Session 4: From Batch to Intraday Operations

In Session 3, the engine was a daily-cadence learning system: EWLS adjusted SIM parameters at each close, an epsilon-greedy bandit learned the CES elasticity per regime, and four formal validation gates produced a go/no-go for production. The engine works. The question now is *how it operates day-to-day on a desk-realistic cadence*.

__Challenge:__ A daily rebalance misses the intraday surface where most news-driven shocks unfold; running the engine on a 30-minute cadence with full auto-execution is reckless because some of the trades it proposes deserve human review; and a 4pm sign-off does not match an intraday operating loop. In this session, we move the engine to a 30-minute cadence, wrap each proposed trade in a two-tier compliance system that splits trades into auto-execute and queue, and structure the human-in-the-loop step as an evening desk-review meeting that signs off on tomorrow's opening posture. Three AI techniques run the loop in production.

* **Online supervised learning (EWLS).** The same recursive least squares estimator with exponential forgetting from Session 3 runs on 30-minute bars unchanged; only the half-life and $\Delta t$ are retuned. Each fire updates one observation's worth of sufficient statistics in constant time, and the calendar-invariant half-life keeps the algorithm's memory in line with what a senior PM would specify.
* **Reinforcement learning for elasticity (epsilon-greedy bandit).** The Session 3 per-regime CES bandit selects $\eta$ at each fire from the regime-local arm policy. The intraday cadence accelerates the bandit's pull rate by $13\times$, which means convergence per regime in days rather than months; the same regret bound applies bar-for-bar.
* **LLM-driven signal extraction with human-in-the-loop oversight.** The hourly news pipeline scores headlines via Claude, computes per-ticker sentiment and severity, and flags tickers whose severity exceeds the desk threshold. Trades on flagged tickers route to the human-review queue regardless of size. The queue plus the evening sign-off is the operational AI safety pattern: the model proposes, the human disposes on the hard cases.

The session treats the engine, the queue, and the evening sign-off as one operating system. Maya runs it, Lou audits it, and the cron schedule is the heartbeat.

> __Learning Objectives:__
>
> By the end of this session, we will be able to:
> * __Run the engine on a 30-minute intraday cadence:__ Convert the EWLS half-life and the per-bar time step from daily to bar units via [`intraday_half_life`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_half_life) and [`intraday_dt`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_dt). Recognize that the EWLS recursion and the bandit update are both bar-interval-agnostic, so the cadence change is a parameter retune, not a redesign.
> * __Split proposed trades into auto-execute and queue:__ Use [`gate_check`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#gate_check) and [`split_intraday_trades`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#split_intraday_trades) so that small low-risk trades clear automatically while large or news-flagged trades route to the human-review queue. Tune the gate thresholds so the queue is non-empty most days, which is where the model's view and the desk's view actually disagree.
> * __Adjudicate the queue and sign tomorrow's ticket:__ Walk a day's [`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem) list and record decisions as [`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision) artifacts with a documented rationale on each one. Commit the next-day allocation as a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket) consumed by tomorrow's 9:35am cron, which closes the human-in-the-loop loop into the production system.

Let's move the engine intraday.

___

## Examples

The example notebooks below accompany this lecture. They split into __core__ examples that we run live alongside the lecture, and __optional__ extension material we can study afterward at our own pace. Before the live run, the [Session 4 Introduction](eCornell-AI-Finance-S4-Introduction-MayasIntradayDesk-May-2026.ipynb) sets the scene at __Lindenfield Wealth Partners__ on Monday May 11, 7pm ET, with today's intraday tape and queue waiting on Maya's desk; we read it first.

### Core (tonight)

These three notebooks carry the session's intraday operating loop: the cron-driven engine that fires every 30 minutes during market hours, the evening desk-review notebook that adjudicates today's queued trades, and the sign-off notebook that commits tomorrow's opening posture as a real next-morning order.

> __Run live with the lecture:__
>
* [▶ Let's walk one fire of the cron-driven intraday engine](eCornell-AI-Finance-S4-Example-Core-ProductionEngine-May-2026.ipynb). We step through a single 30-minute fire of `production_runner.jl`: bar fetch, EWLS update, allocator, gate split into auto-execute and queue, and audit-tape append. Then we loop the same step across all 13 fires of the day to produce the tape the EveningReview notebook reads.
* [▶ Let's run the evening desk review on today's tape](eCornell-AI-Finance-S4-Example-Core-EveningReview-May-2026.ipynb). We open today's intraday tape, score the day on P&L attribution, regime drift, and bandit elasticity, then walk the queue trade-by-trade and record adjudications as [`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision) artifacts.
* [▶ Let's refresh the news, recompute the opening allocation, and sign tomorrow's ticket](eCornell-AI-Finance-S4-Example-Core-TomorrowsTicket-May-2026.ipynb). We trigger a live news refresh in front of the class, run [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket) against tonight's parameters, and commit a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket) that the May 12 9:35am cron submits to Alpaca paper.

### Optional (self-study)

This notebook goes deeper on the news ingestion pipeline that backs the intraday engine. It is not required for the evening desk review.

> __Extension material (self-study):__
>
* [▶ Let's open the AI sentiment pipeline](eCornell-AI-Finance-S4-Example-Optional-AISentimentDeepDive-May-2026.ipynb). We trace one headline through [`MyNewsItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyNewsItem) and [`score_news_with_claude!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#score_news_with_claude!), recover the planted news loadings $\nu_i$ via [`estimate_sim_with_news`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#estimate_sim_with_news), and run a held-out test for overfitting.
>
> Study this when we want to see the mechanics behind the news flag that triggers a queue routing in the Core notebooks.

___

## Notation inherited from prior sessions

This session promotes the validated S3 engine to intraday production operations. The symbols below carry over from S1, S2, and S3 with their definitions intact; we restate the role each plays in the cron loop.

> __Inherited from prior sessions:__
>
> * $g_i, g_{\mathrm{mkt}}, \varepsilon_i$: asset and market growth rates and SIM residuals (S1 [§ From Prices to Growth Rates](../session-1/eCornell-AI-Finance-S1-Lecture-StressTestingMinVariancePortfolios-May-2026.ipynb#from-prices-to-growth-rates)).
> * $\hat{\alpha}_i, \hat{\beta}_i, \hat{\sigma}_{\varepsilon,i}$: EWLS running estimates of the SIM parameters from S3, updated each fire by the recursion in [S3 § EWLS](../session-3/eCornell-AI-Finance-S3-Lecture-OnlineLearningValidation-May-2026.ipynb#ewls-online-sim-parameter-estimation).
> * $\lambda_t, \eta, \eta_{\min}, \eta_{\max}, \gamma_i$: sentiment signal, CES elasticity and bounds, and Cobb-Douglas preference weights (S2 [§ Cobb-Douglas Utility](../session-2/eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb#cobb-douglas-utility) and [§ CES](../session-2/eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb#ces-constant-elasticity-of-substitution)). The intraday $\lambda_t$ is recomputed from the bar-level price buffer rather than from end-of-day closes.
> * $\rho_t \in \{\mathrm{bear}, \mathrm{neutral}, \mathrm{bull}\}$: regime label at fire time $t$, derived from $\lambda_t$ via the threshold $\theta$ defined in [S3 § Per-Regime Eta-Bandit](../session-3/eCornell-AI-Finance-S3-Lecture-OnlineLearningValidation-May-2026.ipynb#applying-the-bandit-to-ces-elasticity).
> * $\mathbf{w}, w_i, n_i$: portfolio weight vector, asset weight, share count (S1 § Notation conventions, S2 § Cobb-Douglas Utility).
> * $\nu_i$: news-sentiment loading coefficient introduced in [S2 § Cobb-Douglas Utility](../session-2/eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb#cobb-douglas-utility); used in this session's news-aware $\gamma_i$ extension.
> * $K, T, \tau_{\max}, d_{\max}$: ticker universe size, horizon, turnover cap, drawdown limit (S2 [§ Rebalancing Engine](../session-2/eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb#the-rebalancing-engine) with intraday reinterpretation below).

> __New in this session:__ $B$ (bars per regular session), $\Delta t = 1/(252 B)$ (per-bar time step), $h_{\text{bars}} = h_{\text{trading days}} \cdot B$ (EWLS half-life in bars), $\mathcal{U}, \mathcal{G}, \mathbf{S}_{\text{intra}}$ (universe, gate config, intraday market price buffer), and the gate thresholds $c_{\max}$, $\text{size}_{\max}$, $s_{\max}$. Each is defined at first use below.


## From Batch to Intraday

In Session 3, the engine fired once per trading day at the close. That cadence is fine for backtesting but does not match how an institutional desk operates: parameters drift through the trading day, news arrives at all hours, and a once-a-day allocation cannot react to a mid-morning regime shift. We move to a **30-minute intraday cadence** for Session 4.

> __Why intraday is a parameter change, not a redesign:__
>
> The EWLS recursion is a recursive least squares estimator with exponential forgetting. Its update rule depends on the *number of observations*, not on the calendar interval between them. The same is true of the epsilon-greedy bandit: pulls update arm means in constant time per observation. Both algorithms are **bar-interval-agnostic**. Moving from daily to 30-minute bars changes only three knobs: the time step $\Delta t$, the EWLS half-life expressed in bars, and the volatility annualization factor.

Let $B$ denote the number of regular-session bars per trading day for a given bar interval, computed from the 6.5-hour US equity session as $B = \lfloor 390 / \text{bar minutes} \rfloor$. The annualized time step and the bar-count half-life follow:

$$\boxed{\Delta t \;=\; \frac{1}{252 \cdot B}, \qquad h_{\text{bars}} \;=\; h_{\text{trading days}} \cdot B \quad\blacksquare}$$

The companion script implements these as [`bars_per_day`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#bars_per_day), [`intraday_dt`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_dt), and [`intraday_half_life`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_half_life). The engine's TOML configuration stores `half_life_trading_days` (a property a senior PM can reason about) and the runner translates it to $h_{\text{bars}}$ at startup, so changing `bar_minutes` in config does not require re-tuning anything else.

This is an *online* algorithm: each cron fire receives the latest bar, updates state, decides, and persists. No batch step. The mechanics are compact enough to state as pseudocode directly.

### Algorithm: Intraday Engine Step (Online)

__Initialize__: Given a ticker universe $\mathcal{U} = \{1, \ldots, K\}$, a bar interval `bar_minutes`, an EWLS state $\{\hat\alpha_i, \hat\beta_i, \hat\sigma_{\varepsilon,i}\}_{i \in \mathcal{U}}$ seeded from a frozen calibration, a compliance gate config $\mathcal{G}$, a rolling intraday market-price buffer $\mathbf{S}_{\text{intra}}$ (reset each trading day), and an Alpaca paper client, set $\Delta t \gets$ [`intraday_dt(bar_minutes)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_dt) and $h_{\text{bars}} \gets$ [`intraday_half_life(h_{\text{trading}}, bar\_minutes)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#intraday_half_life).

On each fire at time $t$ __do__:

1. Fetch new bars since the last seen timestamp; if none, skip.
2. For each new bar at timestamp $t_b$, compute the per-bar growth $g_{m, t_b} \gets \frac{1}{\Delta t} \log(S_{t_b} / S_{t_b - 1})$ and call [`ewls_update!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#ewls_update!) per ticker.
3. Recompute sentiment and effective lambda from $\mathbf{S}_{\text{intra}}$, then bin into a regime $\rho_t \in \{\text{bear}, \text{neutral}, \text{bull}\}$.
4. Run the allocator at the regime-appropriate elasticity $\eta_t$ to obtain target weights $\mathbf{w}^\star_t$.
5. Compute proposed trades $\Delta \mathbf{n}_t$ from current positions, prices, and $\mathbf{w}^\star_t$, with per-trade dollar value and post-trade weight.
6. Join per-ticker news severity from the latest hourly news artifact.
7. Call [`split_intraday_trades`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#split_intraday_trades) with $\Delta \mathbf{n}_t$, the news-severity snapshots, and $\mathcal{G}$ to obtain:
    - $\mathcal{A}_t$: trades that clear all gates (auto-execute via Alpaca).
    - $\mathcal{Q}_t$: trades that fail at least one gate (append to today's queue file as [`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem)).
8. Submit each $a \in \mathcal{A}_t$ to Alpaca paper; append each $q \in \mathcal{Q}_t$ to the queue.
9. Append the fire's audit entry (sentiment, $\lambda$, regime, $\eta$, target weights, $|\mathcal{A}_t|$, $|\mathcal{Q}_t|$) to today's tape.

__Output__: Persist the updated state and exit. The next cron fire picks up from the persisted state.

The same step runs 13 times per trading day. The 16:00 close fire additionally invokes [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket) to write a [`MyTomorrowsTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyTomorrowsTicket) that the evening review notebook will load. With the cadence settled, the next question is what each fire's proposals are allowed to do without human review.

___

## The Two-Mode Cron: Auto-Execute and Queue

Each fire produces a basket of proposed trades. We do not auto-execute every proposal: a trade large enough to move the portfolio's risk profile materially deserves human review even if the engine likes it. The cron splits proposals into two piles via [`split_intraday_trades`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#split_intraday_trades).

> __Per-trade and portfolio-level gates:__
>
> A trade clears the gates and auto-executes when **all** of the following hold; any failure routes the trade to the queue.

| Gate | Scope | Threshold | Why It Exists |
|------|-------|-----------|---------------|
| **Concentration cap** | per-trade | post-trade weight $w_i$ in any single ticker $\le c_{\max}$ | A single-name bet beyond the cap is a portfolio-level decision, not a model decision. |
| **Position size limit** | per-trade | $\lvert\text{trade dollar value}\rvert \le \text{size}_{\max}$ | Caps the dollar magnitude of any single click. Cheap to enforce, cheap to override. |
| **News severity** | per-trade | news-severity score for the ticker $\le s_{\max}$ | If a high-severity headline lands during the fire, the engine cannot judge it. The PM can. |
| **Turnover limit** | portfolio-level | $\sum_i \lvert\text{trade dollar value}_i\rvert \,/\, V_t \le \tau_{\max}$ | Caps per-fire turnover against portfolio value $V_t$. If exceeded, **all** proposed trades route to the queue (not just the largest). |

Threshold values live in the `[Compliance]` section of `production-config.toml`. We tune them so the queue is non-empty most days: too lax and the evening review meeting has nothing to do; too tight and the queue floods and the engine effectively does nothing automatically. The companion examples take a concrete tuning pass.

Trades that clear the gates are submitted via the Alpaca SDK on the fire that produced them. Trades that fail are serialized as [`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem) records (with the engine snapshot stamped in for review context) and appended to today's queue file via [`append_queue_item!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#append_queue_item!). The desk reads the queue at 7pm and works it. The next section covers the third feed into the engine, which is also the third AI surface.

> __Example__
>
> [▶ Let's walk one fire of the cron-driven intraday engine](eCornell-AI-Finance-S4-Example-Core-ProductionEngine-May-2026.ipynb). We step through a single 30-minute fire end-to-end (bar fetch, EWLS update, allocator, gate split into auto-execute and queue, audit-tape append), then loop the same step across all 13 fires of the day to produce the tape the EveningReview notebook reads.

___

## The Hourly News Pipeline

News flow is the third cron on the schedule. Where the engine fires every 30 minutes, the news scorer fires hourly: each fire pulls headlines, scores them via Claude, aggregates per-ticker sentiment and severity, and writes an artifact at `data/news/news-YYYY-MM-DD-HH.jld2`. The next engine fire reads the latest artifact and joins severity onto each proposed trade.

> __Why hourly, not per-bar:__
>
> Calling Claude on every 30-minute engine fire would double the LLM cost and offer little marginal information; mainstream news cycles change on the order of an hour, not a half hour. Hourly cadence keeps the news signal fresh enough to flag a trade in the same fire that an event lands in, while keeping cost predictable.

The source backing the pipeline is configurable via `config/news-source.toml` and supports three modes.

* __Synthetic.__ The cron drives the pipeline from an on-disk corpus generated by [`simulate_news_corpus`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#simulate_news_corpus). The May 5 to May 10 cron operates in this mode so the week's narrative is controlled; the [Optional AISentimentDeepDive notebook](eCornell-AI-Finance-S4-Example-Optional-AISentimentDeepDive-May-2026.ipynb) traces a single synthetic headline through the pipeline.
* __NewsAPI.__ The cron pulls real headlines from a third-party API (NewsAPI.org) and runs [`score_news_with_claude!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#score_news_with_claude!) with `live = true` so each item carries a real LLM score. Requires the `NEWSAPI_KEY` environment variable.
* __Anthropic web fetch.__ The cron uses Anthropic's web fetch tool to pull and score in one call. Requires the `ANTHROPIC_API_KEY` environment variable.

Every news artifact carries the same shape regardless of source: a list of [`MyNewsItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyNewsItem) records with `claude_score` populated, plus per-ticker aggregates `sentiment` and `severity`. The runner's join is identical between synthetic and real modes, so swapping sources is a one-line config change. The class-night demo on May 11 switches to a real-source mode for one fire so the class watches a real headline produce a real sentiment shift in real time. With cadence, gates, and news settled, the day's three artifacts (tape, queue, ticket) are now well-defined; the desk's evening meeting reads them in order.

___

## The Evening Desk Review

By 4pm close, the day has produced three artifacts: an intraday tape (one entry per fire), a queue (one entry per flagged trade), and tomorrow's ticket (the engine's proposed opening allocation). The evening desk review reads all three.

> __What the meeting is for:__
>
> The engine has already executed the easy decisions during the day. The meeting exists to handle the *hard* decisions: the trades whose magnitude, concentration, or news context puts them in territory where a model cannot adjudicate alone. Maya works the queue trade-by-trade with the engine snapshot in front of her, decides approve / reject / modify, records a rationale, and signs each decision. The signed decisions are what the next morning's cron actually does.

Each queue item carries its full decision context: the timestamp, the ticker and side, the gate violations that routed it, and an `engine_snapshot::Dict{String,Any}` with the EWLS state, the sentiment, the lambda, the regime, and the news-source path active at the fire. The meeting's deliverable is a vector of [`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision) records, one per queue item, written to `data/decisions/decisions-YYYY-MM-DD.jld2` via [`save_signed_decisions!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#save_signed_decisions!).

Three adjudication actions are supported.

* __Approve.__ Sign the trade as proposed; the next morning's cron submits the original quantity. Default for trades that queued only because of a portfolio-level turnover cap.
* __Reject.__ Sign the trade as cancelled; the next morning's cron skips it. Default for trades whose only gate violation is high news severity, where the desk does not yet trust the news direction.
* __Modify.__ Sign a new quantity; the next morning's cron submits at the modified size. Default for concentration-cap or position-size violations where the desired direction is fine but the size needs to come down.

The meeting is also where intraday performance gets reviewed: P&L attribution by hour, EWLS estimate drift visualized against the prior-day end-of-day estimates, bandit pulls per regime, and any escalation events the runner logged. Lou audits the rationale field on every decision; an undocumented decision is a finding. With the queue worked, the meeting's last task is committing tomorrow's posture.

> __Example__
>
> [▶ Let's run the evening desk review on today's tape](eCornell-AI-Finance-S4-Example-Core-EveningReview-May-2026.ipynb). We open today's intraday tape, score the day on P&L attribution, regime drift, and bandit elasticity, then walk the queue trade-by-trade and record adjudications as [`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision) artifacts.

___

## Tomorrow's Ticket and Sign-Off

After the queue is worked, attention shifts to tomorrow's opening posture. The 16:00 close fire wrote `data/tickets/ticket-YYYY-MM-DD.jld2` (where the date is the *next* trading day) via [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket). The ticket carries the engine's recommended target weights and the trade list needed to move from today's closing positions to those weights, plus the sentiment snapshot, news flags, the elasticity used, and the regime label.

> __The sign-off step:__
>
> The class refreshes the news pipeline one more time at 7pm against tonight's headlines (the only real-source fire in the week if we are in synthetic mode for May 5 to 10). It then re-runs the allocator with the refreshed sentiment, compares the resulting ticket to the 16:00-generated ticket, and decides whether to commit the original ticket, modify it, or reject specific trades. The committed artifact is a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket) wrapping the original ticket plus any modifications, signed by Maya, persisted via [`save_signed_ticket!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#save_signed_ticket!).

The next-morning 9:35am cron runs `production_runner.jl --mode=execute_signed_ticket`, loads the signed ticket via [`load_signed_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#load_signed_ticket), applies any sign-off modifications, and submits the resulting orders to Alpaca paper. This closes the loop: tonight's human-in-the-loop decision is mechanically committed to the next morning's market open. With the loop closed, the last thing to put on the table is what can still go wrong.

> __Example__
>
> [▶ Let's refresh the news, recompute the opening allocation, and sign tomorrow's ticket](eCornell-AI-Finance-S4-Example-Core-TomorrowsTicket-May-2026.ipynb). We trigger a live news refresh, run [`build_tomorrows_ticket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#build_tomorrows_ticket) against tonight's parameters, and commit a [`MySignedTicket`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedTicket) that the May 12 9:35am cron submits to Alpaca paper.

___

___

## Industry Validation

The intraday operating loop described above (real-time news ingestion, ML/NLP signal extraction, portfolio-level alerts with explainable confidence scores, and human sign-off on suggested actions) matches the construction reported in published industry deployments for portfolio monitoring.

> __Industry analog (anonymized case study):__
>
> A European asset manager reported building an ML/NLP pipeline that scraped thousands of real-time news sources, identified entity linkages across holdings (issuer, supplier, counterparty), assessed event severity, and generated portfolio-level alerts with suggested actions. Reported impact: 15% reduction in drawdowns on monitored portfolios, with detection of material events hours faster than legacy systems. Their published lesson on what made the system trusted by regulators and PMs alike: top contributing signals, confidence scores, and natural-language explanations exposed for every alert. This is the explainability triumvirate the production engine in this session is built around. The synthetic news corpus and Claude scoring path teach the methodology; the audit-log and confidence-score fields surfaced in the evening review are how the system stays defensible.

## Risks and Limits

The intraday loop closes a real human-machine cycle, but several failure modes deserve explicit treatment before we hand the engine the keys.

* __Overnight gaps.__ A signed ticket sized at last night's close prices submits at this morning's open prices. If the underlying gaps overnight (earnings, macro print, geopolitical event), the realized fill differs from the engine's intent. The compliance queue does not see overnight risk; the position-size and concentration gates do, but only on the *signed* quantities, not on the gap-adjusted ones. Mitigation lives in setting the position-size limit small enough that an overnight gap on any single name is bounded.
* __News API outage at 7pm.__ If the real-source news fire fails (rate limit, network, vendor downtime), the sign-off notebook falls back to the synthetic corpus rather than blocking the meeting. The fallback path is documented in the TomorrowsTicket notebook; the lecture is honest that *some* live demos are not actually live.
* __Stale EWLS state on long pauses.__ The EWLS half-life decays bar-by-bar, not in calendar time, so weekends and holidays pass without decay because no bars arrive. That is fine for a regular two-day weekend or a three-day weekend. But if the cron is paused for a full week (vacation, infrastructure work), the next fire fetches a long bar history and the EWLS estimates absorb a week of update steps in a single fire and can move materially. We document a rule of thumb: any pause longer than a week, re-seed EWLS from a fresh calibration before resuming.
* __Compliance gate hyperparameter drift.__ Tuning the gates so the queue is non-empty most days is a live calibration, not a one-time setting. As the portfolio grows, the position-size limit and concentration cap need to scale, or the queue empties and the meeting becomes a rubber-stamp.

These are the lecture's scope. Production deployment beyond paper trading invites additional concerns (regulatory reporting, broker outages, fat-finger protections, time-of-day liquidity bias) that we leave to follow-up engagement.

___

## Summary

In this session, we moved the engine from a daily backtest cadence to a 30-minute intraday operating loop, wrapped each proposal in a two-tier compliance system, and structured the human-in-the-loop step as an evening desk-review meeting that signs off on the next morning's open.

> __Key Takeaways:__
>
> * __EWLS and the bandit are bar-interval-agnostic:__ The same online estimators that powered the daily-cadence engine in Session 3 run on 30-minute bars unchanged; only $\Delta t$, the half-life unit, and volatility annualization need to be retuned. We can move the cadence faster or slower without redesigning the algorithm.
> * __The queue is where AI safety lives:__ Auto-executing every proposal is reckless; routing every proposal to a human is paralysis. Tightening the gates so the queue is non-empty most days produces a steady stream of decisions where the model's view and the PM's view actually diverge, and the meeting is where that disagreement gets adjudicated.
> * __The signed ticket closes the loop:__ Tonight's sign-off becomes tomorrow's submitted order. The evening review is not a documentation exercise; it is the production system's commit step, and the next morning's cron is what makes it real.

This is the final session of the course. The full pipeline (SIM estimation in Session 1, adaptive allocation in Session 2, online learning and validation in Session 3, intraday production with evening sign-off in Session 4) is now end-to-end.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The production system described is a pedagogical tool using paper trading and synthetic news. Real-world deployment requires regulatory approval, compliance review, operational infrastructure, and risk management beyond the scope of this course.

___